<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## Neighborhoods and Search Comparisons

**Explain why local optimality depends on the allowed moves and compare methods on the same problem.**

Keep the five-city symmetric distance matrix, the requirement to visit every city once and return, and the total-distance objective. The decision is a discrete visit order. The clock observations, calibration equality, and correction bound do not apply to this delivery system. The controlled neighborhood comparison keeps the current tour fixed. The method comparison keeps the distance matrix and feasibility requirements fixed.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 1 · Choose a complete tour

A vehicle must visit cities $0,1,2,3,4$ once each and return to its starting city. A **tour** is that complete closed route. Shorter total distance means less travel.

Let $C_{ij}$ be the fixed travel distance from city $i$ to city $j$. The source gives no physical unit, so we report **distance units**. All city pairs are connected, and the distances are symmetric: $C_{ij}=C_{ji}$. The numbers are travel costs; we do not assume they are straight-line distances on a map.

| From / to | $0$ | $1$ | $2$ | $3$ | $4$ |
|:---|---:|---:|---:|---:|---:|
| $0$ | 0 | 2 | 9 | 10 | 7 |
| $1$ | 2 | 0 | 8 | 5 | 6 |
| $2$ | 9 | 8 | 0 | 4 | 3 |
| $3$ | 10 | 5 | 4 | 0 | 1 |
| $4$ | 7 | 6 | 3 | 1 | 0 |

Write the decision as $x=(v_0,v_1,v_2,v_3,v_4,v_5)$, where $v_k$ is the city visited at position $k$. Feasibility requires that $v_0,\ldots,v_4$ contain every city exactly once and $v_5=v_0$. The objective is

$$
f(x)=\sum_{k=0}^{4} C_{v_k,v_{k+1}}.
$$

The final term includes the return trip. For example,

$$
f(2,4,3,1,0,2)=3+1+5+2+9=20.
$$

The distance matrix is fixed data. The visit order is the physical decision. Here the raw performance output, total distance, is also the scalar objective. A route with a missing city or return trip is rejected before any score comparison.


In [ ]:
import numpy as np

# Fixed travel distances, copied from the supplied example_1.py
DISTANCES = np.array([
    [0, 2, 9, 10, 7],
    [2, 0, 8, 5, 6],
    [9, 8, 0, 4, 3],
    [10, 5, 4, 0, 1],
    [7, 6, 3, 1, 0],
], dtype=int)
CITY_COUNT = len(DISTANCES)


def evaluate_tour(tour):
    tour = tuple(tour)
    valid_labels = all(isinstance(city, (int, np.integer))
                       and 0 <= city < CITY_COUNT for city in tour)
    feasible = (len(tour) == CITY_COUNT + 1 and valid_labels
                and tour[0] == tour[-1]
                and set(tour[:-1]) == set(range(CITY_COUNT)))
    if not feasible:
        return {"decision": tour, "feasible": False,
                "edge_distances": None, "objective": None}
    edges = np.array([DISTANCES[a, b] for a, b in zip(tour, tour[1:])])
    return {"decision": tour, "feasible": True,
            "edge_distances": edges, "objective": int(edges.sum())}


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 2 · Define complete-tour improvement

A neighborhood $N(x)$ contains the tours reachable from a current tour $x$ by one allowed move. A **2-opt move** reverses internal positions $i$ through $j$ while preserving the first and final city. Here $1\le i<j\le4$; reversing all four internal positions is excluded because it only reverses the same undirected cycle. The remaining five moves preserve feasibility.

**Best-improvement local search** evaluates all neighbors, accepts the shortest one only when it strictly improves the current distance, and repeats. A complete scan without improvement establishes $f(x)\le f(x')$ for every $x'\in N(x)$. That is a statement about the chosen neighborhood, not all feasible tours.


In [ ]:
def two_opt_neighbors(tour):
    tour = tuple(tour)
    city_count = len(tour) - 1
    for i in range(1, city_count - 1):
        for j in range(i + 1, city_count):
            if i == 1 and j == city_count - 1:
                continue  # Same undirected cycle, written in reverse.
            candidate = tour[:i] + tour[i:j + 1][::-1] + tour[j + 1:]
            yield (i, j), candidate


In [ ]:
def local_search(initial_tour, neighborhood=two_opt_neighbors):
    current = evaluate_tour(initial_tour)
    if not current["feasible"]:
        raise ValueError("Local search needs a complete feasible initial tour.")
    history = [current]
    accepted_moves = []
    neighbor_evaluations = 0
    while True:
        best = current
        best_move = None
        for move, tour in neighborhood(current["decision"]):
            candidate = evaluate_tour(tour)
            neighbor_evaluations += 1
            if not candidate["feasible"]:
                continue
            if candidate["objective"] < best["objective"]:
                best, best_move = candidate, move
        if best_move is None:
            return {"result": current, "history": history,
                    "accepted_moves": accepted_moves,
                    "neighbor_evaluations": neighbor_evaluations,
                    "stop_reason": "No strictly improving neighbor"}
        current = best
        history.append(current)
        accepted_moves.append(best_move)


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 3 · A local optimum depends on the neighborhood

Keep the matrix, objective, and current tour fixed at $x=(2,3,1,0,4,2)$, with distance 21. Change only which moves the search may consider.

First allow swaps of consecutive internal positions: $(1,2)$, $(2,3)$, or $(3,4)$. The initial and final city remain fixed. This deliberately restricted neighborhood produces distances 33, 25, and 31, so the tour is a local optimum under this rule.

Now allow the full 2-opt neighborhood. Reversing positions 1 through 3 gives $(2,0,1,3,4,2)$, with distance 20. The old tour was locally optimal for the smaller neighborhood and was not globally optimal.

Each card below shows a complete tour's total distance, not a city label. Compare the same current distance of 21 under the two move rules. All three adjacent-swap neighbors are worse; the lower panel shows one improving 2-opt neighbor.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/05-1_heuristic_optimization_greedy_and_local_search/assets/local_optimum_neighborhood-v4.png" alt="With current tour distance 21, the three adjacent-position swaps give distances 33, 25, and 31. Allowing 2-opt adds an improving neighbor with distance 20, so local optimality depends on the allowed moves." width="512" style="display: block; max-width: 100%; height: auto; margin: 0; background-color: #fff;">
</div>

Only the neighbor generator changes. The same feasibility checks, score, and improvement rule apply to both runs.


In [ ]:
def adjacent_neighbors(tour):
    tour = tuple(tour)
    for i in range(1, len(tour) - 2):
        candidate = list(tour)
        candidate[i], candidate[i + 1] = candidate[i + 1], candidate[i]
        yield (i, i + 1), tuple(candidate)


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

Verify the restricted local optimum and the improvement permitted by 2-opt. This example does not establish that 2-opt always reaches a global optimum; it shows why every local-optimality statement must name its neighborhood.


In [ ]:
comparison_tour = (2, 3, 1, 0, 4, 2)
print("Consecutive-position neighbor distances:",
      [evaluate_tour(tour)["objective"]
       for _, tour in adjacent_neighbors(comparison_tour)])
restricted_run = local_search(comparison_tour, adjacent_neighbors)
wider_run = local_search(comparison_tour, two_opt_neighbors)
for name, run in (("Consecutive positions", restricted_run), ("2-opt", wider_run)):
    print(name, "distance:", run["result"]["objective"],
          "accepted moves:", run["accepted_moves"])


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 4 · Establish construction and exact references

Nearest-neighbor construction repeatedly chooses the closest unvisited city, breaks ties by smaller city label, and returns to its starting city after all visits. No depot or departure city is imposed, so the construction start is a search hyperparameter; rotating a fixed cycle changes neither its edges nor its distance.

The definitions below also enumerate every ordering with city 2 fixed as the written start. The $4!=24$ written tours include reverse orientations. This complete enumeration verifies the reference distance 20 independently of local search.


In [ ]:
def nearest_neighbor(start_city):
    if start_city not in range(CITY_COUNT):
        raise ValueError("The starting city must be one of the city labels.")
    remaining = set(range(CITY_COUNT)) - {start_city}
    tour = [start_city]
    trace = []
    while remaining:
        current = tour[-1]
        next_city = min(remaining,
                        key=lambda city: (DISTANCES[current, city], city))
        trace.append({"current": current, "unvisited": tuple(sorted(remaining)),
                      "chosen": next_city,
                      "distance": int(DISTANCES[current, next_city])})
        tour.append(next_city)
        remaining.remove(next_city)
    tour.append(start_city)
    result = evaluate_tour(tour)
    result["trace"] = trace
    return result


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

The calculation creates one greedy tour from each start, verifies the optimum by complete enumeration, and records the local-search sequence from start 1.


In [ ]:
greedy_results = [nearest_neighbor(start) for start in range(CITY_COUNT)]
from itertools import permutations

reference_start = 2
other_cities = [city for city in range(CITY_COUNT) if city != reference_start]
enumerated_results = [
    evaluate_tour((reference_start,) + order + (reference_start,))
    for order in permutations(other_cities)
]
exact_result = min(enumerated_results, key=lambda result: result["objective"])
exact_distance = exact_result["objective"]
print("Complete tours evaluated:", len(enumerated_results))
print("One globally optimal tour:", exact_result["decision"])
print("Verified optimal distance:", exact_distance)

initial_tour = greedy_results[1]["decision"]
improved_run = local_search(initial_tour)


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 5 · Compare results while keeping the problem fixed

Apply the same 2-opt search to every greedy result. Compare the initial distance with the final distance for each construction start. In this instance all five searches reach 20; full enumeration independently verifies that value.

The upper plot shows how the construction start changes greedy quality. The lower plot shows the accepted improvements from start 1. The dashed line is the optimum verified by exhaustive enumeration, not an estimate from the plotted runs.


In [ ]:
import matplotlib.pyplot as plt

BLUE = "#2878b5"
TEAL = "#168578"
ORANGE = "#e78b24"
GRAY = "#707070"

all_improved = [local_search(result["decision"]) for result in greedy_results]


def show_search_comparison():
    figure, axes = plt.subplots(2, 1, figsize=(5.6, 7.0))
    starts = np.arange(CITY_COUNT)
    greedy_distances = [result["objective"] for result in greedy_results]
    final_distances = [run["result"]["objective"] for run in all_improved]
    axes[0].bar(starts - 0.18, greedy_distances, width=0.36,
                color=BLUE, label="Greedy tour")
    axes[0].bar(starts + 0.18, final_distances, width=0.36,
                color=TEAL, label="After 2-opt")
    axes[0].set(xticks=starts, xlabel="Construction start (city label)",
                ylabel="Total distance (distance units)", ylim=(0, 28),
                title="The start changes greedy tour quality")
    axes[0].legend(loc="lower left", fontsize=10)

    history = [result["objective"] for result in improved_run["history"]]
    steps = np.arange(len(history))
    axes[1].plot(steps, history, marker="o", color=ORANGE,
                 linewidth=2, label="Current tour from start 1")
    axes[1].set(xticks=steps, xlabel="Number of accepted 2-opt moves",
                ylabel="Total distance (distance units)", ylim=(19.5, 23),
                title="Two accepted moves reduce 22 to 20")
    for step, distance in zip(steps, history):
        axes[1].annotate(str(distance), (step, distance), xytext=(0, 8),
                         textcoords="offset points", ha="center")
    for axis in axes:
        axis.axhline(exact_distance, color=GRAY, linestyle="--",
                     label="Verified optimum: 20", zorder=0)
        axis.grid(alpha=0.25)
        axis.set_axisbelow(True)
    axes[1].legend(loc="upper right", fontsize=10)
    figure.tight_layout()
    return figure


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

Draw the two comparisons from the evaluation records. No map coordinates are invented: every distance still comes from the supplied matrix.


In [ ]:
comparison_figure = show_search_comparison()
plt.show()


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

Enumeration checks $(n-1)!$ written tours after fixing the start. For $n=5$ that is 24; for $n=10$ it is $9!=362{,}880$. A nearest-neighbor construction compares $4+3+2+1=10$ outgoing edge distances here. A full 2-opt scan considers five neighboring tours; the start-1 run uses three scans, including the final check.

These count different operations, so they are not timing measurements. Our code recalculates each neighbor's full distance for clarity. Heuristics can avoid enumerating every tour, but neither a small search effort nor successful runs on this instance prove a general quality guarantee.

| Method | What it does here | What supports its conclusion |
|:---|:---|:---|
| Greedy construction | Adds the nearest unvisited city | Produces a feasible tour; does not certify global optimality |
| Local search | Replaces a full tour with a strictly better neighbor | A complete final scan verifies local optimality for that neighborhood |
| Exhaustive enumeration | Evaluates all 24 written tours | Proves the global optimum for this finite instance |

Greedy rules can be exact for problems with additional structure and a proof. Here nearest neighbor is a heuristic. “Greedy” describes the choice rule; “heuristic” describes the absence of a general optimality guarantee for this application.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 6 · Source material and method reference

The supplied `Lecture_10_1_Heuristic_Optimization/10_1.pptx` introduces heuristic construction rules. Its `ExampleCode_10_1/example_1.py` supplies the five-city distance matrix, nearest-neighbor rule, and original start at city 2. This notebook adds the restricted-neighborhood comparison and compares construction, local search, and exact verification.

Michael A. Trick's [Heuristics](https://mat.tepper.cmu.edu/orclass/integer/node17.html) describes nearest-neighbor tour construction and two-opt tour improvement. The worked calculations use the supplied matrix.
